In [ ]:
!pip install geopandas rasterio cartopy matplotlib shapely geodatasets folium mapclassify

## Geopandas

### Imports

In [2]:
import geopandas as gpd
import rasterio
import matplotlib.pyplot as plt
from shapely.geometry import Point, Polygon

import matplotlib.pyplot as plt
import geopandas
from cartopy import crs as ccrs
import geodatasets
from geodatasets import get_path

### Visualizing

In [ ]:
geodatasets.data

In [4]:
path = get_path("naturalearth.land")
df = gpd.read_file(path)

In [ ]:
df

In [ ]:
df.plot()

In [ ]:
# Define the CartoPy CRS object.
crs = ccrs.AzimuthalEquidistant()

# This can be converted into a `proj4` string/dict compatible with GeoPandas
crs_proj4 = crs.proj4_init
df_ae = df.to_crs(crs_proj4)

# Here's what the plot looks like in GeoPandas
df_ae.plot()

In [ ]:
crs_new = ccrs.Geostationary(
    central_longitude=10.0, satellite_height=35785831,
)
new_geometries = [
    crs_new.project_geometry(ii, src_crs=crs) for ii in df_ae["geometry"].values
]

fig, ax = plt.subplots(subplot_kw={"projection": crs_new})
ax.add_geometries(new_geometries, crs=crs_new)

Other projections at https://scitools.org.uk/cartopy/docs/v0.15/crs/projections.html

Visualizations at https://vcg.isti.cnr.it/~tarini/spinnableworldmaps/index.html

In [ ]:
df_aea = geopandas.GeoDataFrame(
    df.drop(columns="geometry"), geometry=new_geometries, crs=crs_new.proj4_init
)
df_aea.plot()

In [ ]:
# Generate a CartoPy figure and add the countries to it
fig, ax = plt.subplots(subplot_kw={"projection": crs_new})
ax.add_geometries(new_geometries, crs=crs_new)

# Calculate centroids and plot
df_aea_centroids = df_aea.geometry.centroid
# Need to provide "zorder" to ensure the points are plotted above the polygons
df_aea_centroids.plot(ax=ax, markersize=5, color="r", zorder=10)
plt.show()

## How to read files

In [ ]:
# gdf = gpd.read_file(path_to_file)
file_location = geodatasets.get_path("nybb")
file_location

In [12]:
gdf = geopandas.read_file(file_location)

In [ ]:
gdf

#### Filter by bbox

In [14]:
bbox = (
    1031051.7879884212, 224272.49231459625, 1047224.3104931959, 244317.30894023244
)
gdf = geopandas.read_file(
    geodatasets.get_path("nybb"),
    bbox=bbox,
)

#### Filter by feature

In [15]:
gdf = geopandas.read_file(
    geodatasets.get_path("geoda.nyc"),
    columns=["name", "rent2008", "kids2000"],
)

In [ ]:
gdf = geopandas.read_file(
    geodatasets.get_path("geoda.nyc"),
    where="subborough='Coney Island'",
)

gdf

### Plotting

In [ ]:
chicago = geopandas.read_file(geodatasets.get_path("geoda.chicago_commpop"))

groceries = geopandas.read_file(geodatasets.get_path("geoda.groceries"))

chicago.plot();

In [ ]:
chicago.boundary.plot()

In [ ]:
chicago.plot(
    column="POP2010",
    legend=True,
    legend_kwds={"label": "Population in 2010", "orientation": "horizontal"},
    cmap='OrRd',
);

In [ ]:
groceries.plot(marker='*', color='green', markersize=5);
groceries = groceries.to_crs(chicago.crs)

In [ ]:
base = chicago.plot(color='white', edgecolor='black')
groceries.plot(ax=base, marker='o', color='red', markersize=5);

In [ ]:
import folium

m = chicago.explore(
    column="POP2010",  # make choropleth based on "POP2010" column
    scheme="naturalbreaks",  # use mapclassify's natural breaks scheme
    legend=True,  # show legend
    k=10,  # use 10 bins
    tooltip=False,  # hide tooltip
    popup=["POP2010", "POP2000"],  # show popup (on-click)
    legend_kwds=dict(colorbar=False),  # do not use colorbar
    name="chicago",  # name of the layer in the map
)

groceries.explore(
    m=m,  # pass the map object
    color="red",  # use red color on all points
    marker_kwds=dict(radius=5, fill=True),  # make marker radius 10px with fill
    tooltip="Address",  # show "name" column in the tooltip
    tooltip_kwds=dict(labels=False),  # do not show column label in the tooltip
    name="groceries",  # name of the layer in the map
)

folium.TileLayer("CartoDB positron", show=False).add_to(
    m
)  # use folium to add alternative tiles
folium.LayerControl().add_to(m)  # use folium to add layer control

m  # show map

### How to write

In [23]:
from pathlib import Path

Path('data').mkdir(exist_ok=True)

gdf.to_file("data/data.shp")

In [24]:
gdf.to_file("data/data.geojson", driver='GeoJSON')

### Some operations

#### Calculate area

In [25]:
chicago = geopandas.read_file(geodatasets.get_path("geoda.chicago_commpop"))

In [ ]:
chicago.crs

In [ ]:
chicago.area

In [ ]:
chicago_m = chicago.to_crs('EPSG:3857')
chicago_m.area

#### Check if geomatry containts ROI

In [29]:
p = Point(-87.61868, 41.83512)

In [ ]:
chicago.contains(p).any()

#### Clip the data

In [31]:
from shapely import box


polygon = box(-87.8, 41.90, -87.5, 42)


In [ ]:
chicago_clipped = chicago.clip(polygon)

fig, ax = plt.subplots(figsize=(12, 8))
chicago_clipped.plot(ax=ax)
chicago.boundary.plot(ax=ax)
chicago.boundary.plot(ax=ax)
ax.set_title("Chicago Clipped")
ax.set_axis_off()
plt.show()

In [ ]:
chicago_clipped